Définition de la structure du schéma, c'est à dire des colonnes du fichier dbip-country car ce dernier n'a pas d'en-tête.

In [0]:
from pyspark.sql.types import *

dbip_schema = StructType([
  StructField('ip_start', StringType(), True),
  StructField('ip_end', StringType(), True),
  StructField('code_country', StringType(), True)])


Lecture du fichier et chargement dans un dataframe pyspark avec le schéma personnalisé

In [0]:
dbip_df = spark.read.csv('/Volumes/tpnetworkintrusion/date732/rawcsvfiles/dbip-country-lite-2026-01.csv', schema=dbip_schema)

In [0]:
dbip_df.printSchema()

Conserver dans le dataframe dbip uniquement les intervalles ipv4

In [0]:
dbip_df_ipv4 = dbip_df.filter(dbip_df.ip_start.contains('.'))


In [0]:
print(dbip_df.count())
print(dbip_df_ipv4.count())

In [0]:
dbip_df_ipv4.show(10)

Lecture du fichier public_network et chargement en mémoire en pyspark dataframe en utilisant la première ligne comme en-tête des colonnes.

In [0]:
public_network_df = spark.read.csv('/Volumes/tpnetworkintrusion/date732/rawcsvfiles/public_network_logs.csv', header=True)

In [0]:
public_network_df.printSchema()

Réduction du nombre de colonnes utiles à seulement Destination_IP, Intrusion, Source_IP et Scan_Type.

In [0]:
public_network_df_select = public_network_df.select("Destination_IP", "Intrusion", "Source_IP", "Scan_Type")

Appliquer la conversion en entier des colonnes contenant une adresse IPv4 (en chaine de caractère)

In [0]:
import ipaddress

In [0]:
ip_str = '0.0.0.1'
int(ipaddress.ip_address(ip_str))

Définition d'une [fonction UDF](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.udf.html) pour convertir les IPs

In [0]:
@udf(returnType=LongType())
def ip2int(ip_str):
  return int(ipaddress.ip_address(ip_str))

In [0]:
public_network_df_select.select("Destination_IP", ip2int("Destination_IP")).show(5)

Convertir les colonnes ipv4 en valeurs numériques en base10 avec la fonction udf ip2int

In [0]:
dbip_df_num = dbip_df_ipv4.select("ip_start", ip2int("ip_start").alias("ip_start_num"), "ip_end", ip2int("ip_end").alias("ip_end_num"), "code_country")

In [0]:
dbip_df_num.printSchema()

In [0]:
public_network_df_num = public_network_df_select.select("Destination_IP", ip2int("Destination_IP").alias("Destination_IP_num"), "Intrusion", "Source_IP", ip2int("Source_IP").alias("Source_IP_num"), "Scan_Type")


In [0]:
public_network_df_num.printSchema()

Effectuer la jointure pour déterminer le pays source et le pays cible

**Version SQL**

Convertir en tables temporaires les 2 dataframes

In [0]:
public_network_df_num.createOrReplaceTempView("public_network_df_num")
dbip_df_num.createOrReplaceTempView("dbip_df_num")

In [0]:
%sql
select * from public_network_df_num

In [0]:
%sql
select * from dbip_df_num

In [0]:
%sql
select code_country, intrusion, count(intrusion) from public_network_df_num left join dbip_df_num on public_network_df_num.Source_IP_num >= dbip_df_num.ip_start_num and public_network_df_num.Source_IP_num <=  dbip_df_num.ip_end_num group by code_country, intrusion

**Version Python**
Uniquement Spark Dataframe

In [0]:
# public_network_df_num
# dbip_df_num
# join both dataframes using Source_IP_num

result_df = public_network_df_num.join(dbip_df_num, (public_network_df_num.Source_IP_num >= dbip_df_num.ip_start_num) & (public_network_df_num.Source_IP_num <= dbip_df_num.ip_end_num), "inner")

In [0]:

result_df_count = result_df.groupBy("code_country", "Intrusion").count()

In [0]:
result_df_count.show()

**Version Python**
Utilisation des tables Delta

In [0]:
# save both dataframes as delta tables
public_network_df_num.write.mode('overwrite').saveAsTable('tpnetworkintrusion.date732.public_network_num')


In [0]:
dbip_df_num.write.mode('overwrite').saveAsTable('tpnetworkintrusion.date732.dbip_num')

In [0]:
public_network_delta = spark.read.table('tpnetworkintrusion.date732.public_network_num')
dbip_delta = spark.read.table('tpnetworkintrusion.date732.dbip_num')

In [0]:
result_df = public_network_delta.join(dbip_delta, (public_network_delta.Source_IP_num >= dbip_delta.ip_start_num) & (public_network_delta.Source_IP_num <= dbip_delta.ip_end_num), "inner")

In [0]:
result_df_count = result_df.groupBy("code_country", "Intrusion").count()
result_df_count.show()

In [0]:
# joins both delta tables on Source_IP_num and ip_start_num and ip_end_num


Reste à faire

* Travailler uniquement sur les dataframe (Spark) sans passer par les vues temporaires
* Travailler avec la création de tables delta et effectuer la jointure sur les tables delta
* Travailler en mémoire uniquement avec Pandas

In [0]:
import pandas as pd

In [0]:
# read the files thanks to pd.read_csv
# prepare the function used to convert ip into decimal values
# apply to columns ip_destination and ip_start and ip_end
# join both pandas  dataframe and compare results performance

In [0]:
pd.read_csv(, names = [])

In [0]:
import polars as pl